# Proyecto 2
## Agile Community Rules Classification
### Análisis Exploratorio


Intregantes:
- André pivaral  
- Belén Monterroso 
- Melisa Mendizabal 
- Renato Rojas 


# Descripción de los datos y limpieza de datos
### Librerías necesarias

In [20]:
import os
import re
import string
import warnings
 
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
 
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
 
from wordcloud import WordCloud
from langdetect import detect, LangDetectException
 
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

### Set de datos

In [21]:
import pandas as pd

train = pd.read_csv('Datos/train.csv')
test = pd.read_csv('Datos/test.csv')

print(test.head())

   row_id                                               body  \
0    2029  NEW RAP GROUP 17. CHECK US OUT https://soundcl...   
1    2030  Make your life comfortable. Get up to 15% Disc...   
2    2031  Kickin' ass and selling underwear!\nJust made ...   
3    2032   watch  hooters  best  therein  http://clickan...   
4    2033   bitches  for free  at this point  show all  h...   

                                                rule        subreddit  \
0  No Advertising: Spam, referral links, unsolici...      hiphopheads   
1  No legal advice: Do not offer or request legal...        AskReddit   
2  No Advertising: Spam, referral links, unsolici...         gonewild   
3  No Advertising: Spam, referral links, unsolici...  personalfinance   
4  No Advertising: Spam, referral links, unsolici...   Showerthoughts   

                                  positive_example_1  \
0  Hey, guys, just wanted to drop in and invite y...   
1  Get a lawyer and get the security camera foota...   
2  Good 

## Dimensiones de los data sets y variables

In [22]:
print(f"train.csv: {train.shape[0]} filas x {train.shape[1]} columnas")
print(f"test.csv : {test.shape[0]} filas x {test.shape[1]} columnas")
print("\nColumnas train:", list(train.columns))
print("Columnas test :", list(test.columns))

train.csv: 2029 filas x 9 columnas
test.csv : 10 filas x 8 columnas

Columnas train: ['row_id', 'body', 'rule', 'subreddit', 'positive_example_1', 'positive_example_2', 'negative_example_1', 'negative_example_2', 'rule_violation']
Columnas test : ['row_id', 'body', 'rule', 'subreddit', 'positive_example_1', 'positive_example_2', 'negative_example_1', 'negative_example_2']


## Separación de la variable objetivo en test

In [23]:
TARGET = "rule_violation"
assert TARGET in train.columns and TARGET not in test.columns, (
    "Se esperaba que 'rule_violation' esté solo en train (es el target)."
)

## Descripción de las variables
- row_id: numérica discreta (identificador único, no  predictiva)
- body: texto (comentario de Reddit a clasificar)
- rule: categórica (texto de la regla de moderación)
- subreddit: categórica (comunidad de origen)
- positive_example_1/2: texto (ejemplos que sí violan la regla, dados como contexto)
- negative_example_1/2: texto (ejemplos que no violan la regla, dados como contexto)
- rule_violation: categórica binaria (TARGET): 0 = no viola, 1 = viola

### Tipos de variables

In [24]:
print(train.dtypes)

row_id                 int64
body                  object
rule                  object
subreddit             object
positive_example_1    object
positive_example_2    object
negative_example_1    object
negative_example_2    object
rule_violation         int64
dtype: object


## Valores faltantes o presencia de datos nulos
Al no encontrar valores nulos en ninguno de los dos datasets no se aplicará ninguna estrategia de imputación. Esto indica que el dataset presenta un limpieza previa por los organizadores del reto previo a su publicación

In [25]:
nulls_train = train.isnull().sum()
nulls_test = test.isnull().sum()
print("Nulos por columna en train:")
print(nulls_train)
print("\nNulos por columna en test:")
print(nulls_test)
 
total_nulls = nulls_train.sum() + nulls_test.sum()
if total_nulls == 0:
    print(
        "No hay valores nulos."
    )
else:
    print(
        "Sí se encontraron nulos"
    )

Nulos por columna en train:
row_id                0
body                  0
rule                  0
subreddit             0
positive_example_1    0
positive_example_2    0
negative_example_1    0
negative_example_2    0
rule_violation        0
dtype: int64

Nulos por columna en test:
row_id                0
body                  0
rule                  0
subreddit             0
positive_example_1    0
positive_example_2    0
negative_example_1    0
negative_example_2    0
dtype: int64
No hay valores nulos.


## Valores duplicados

### Duplicados exactos

In [26]:
dup_full_train = train.duplicated().sum()
dup_full_test = test.duplicated().sum()
print(f"Filas 100% duplicadas en train: {dup_full_train}")
print(f"Filas 100% duplicadas en test : {dup_full_test}")

Filas 100% duplicadas en train: 0
Filas 100% duplicadas en test : 0


### Duplicados de 'body' 
El mismo comentario aparece varias veces, pero posiblemente evaluado contra reglas/subreddits distintos

In [27]:
dup_body_mask_train = train.duplicated(subset=["body"], keep=False)
n_dup_body_train = train.loc[dup_body_mask_train, "body"].nunique()
print(f"\nComentarios ('body') que se repiten al menos 2 veces en train: "
      f"{n_dup_body_train} textos distintos, "
      f"{dup_body_mask_train.sum()} filas involucradas.")


Comentarios ('body') que se repiten al menos 2 veces en train: 57 textos distintos, 217 filas involucradas.


### Validación de duplicados y tratamiento

NO eliminamos los 'body' duplicados. La unidad de análisis de este problema no es el comentario por sí solo, sino el par (comentario, regla). Un mismo comentario puede violar la regla de 'No Advertising' en un subreddit y no violar la regla de 'No legal advice' en otro contexto, por lo que cada fila aporta información distinta al modelo. Sí eliminamos, en cambio, duplicados exactos de FILA COMPLETA (idéntica en todas las columnas), ya que esos sí son redundancia pura.

In [28]:
dup_groups = (
    train[dup_body_mask_train]
    .groupby("body")[TARGET]
    .nunique()
)
n_consistent = (dup_groups == 1).sum()
n_inconsistent = (dup_groups > 1).sum()
print(f"  - Grupos de duplicados con etiqueta CONSISTENTE: {n_consistent}")
print(f"  - Grupos de duplicados con etiqueta INCONSISTENTE (mismo texto, "
      f"distinta regla -> distinto veredicto): {n_inconsistent}")
 
print("")
before = len(train)
train = train.drop_duplicates().reset_index(drop=True)
after = len(train)
print(f"Filas eliminadas por duplicado exacto de fila completa: {before - after}")
 
before_t = len(test)
test = test.drop_duplicates().reset_index(drop=True)
after_t = len(test)
print(f"Filas eliminadas por duplicado exacto en test: {before_t - after_t}")

  - Grupos de duplicados con etiqueta CONSISTENTE: 43
  - Grupos de duplicados con etiqueta INCONSISTENTE (mismo texto, distinta regla -> distinto veredicto): 14

Filas eliminadas por duplicado exacto de fila completa: 0
Filas eliminadas por duplicado exacto en test: 0


## LIMPIEZA Y NORMALIZACIÓN DE TEXTO

In [29]:
URL_REGEX = re.compile(r"(https?://\S+|www\.\S+)")
MD_LINK_REGEX = re.compile(r"\[([^\]]*)\]\((https?://[^\)]+)\)")  # [texto](url)
MD_BOLD_ITALIC_REGEX = re.compile(r"(\*\*|\*|__|_)")
MD_QUOTE_REGEX = re.compile(r"^>.*$", flags=re.MULTILINE)
EMOJI_REGEX = re.compile(
    "["
    "\U0001F300-\U0001FAFF"  # símbolos, emoticones, transporte, etc.
    "\U00002600-\U000027BF"  # símbolos varios / dingbats
    "\U0001F1E6-\U0001F1FF"  # banderas
    "]+",
    flags=re.UNICODE,
)
NON_ALPHA_REGEX = re.compile(r"[^a-z\s]")

### Cantidad de urls

In [30]:
def count_urls(text):
    raw_urls = len(URL_REGEX.findall(text))
    md_urls = len(MD_LINK_REGEX.findall(text))
    return raw_urls + md_urls

### Cantidad de signos de exclamación

In [31]:
def count_exclamations(text):
    return text.count("!")

### Cantidad de emojis

In [32]:
def count_emojis(text):
    return len(EMOJI_REGEX.findall(text))
